# [001] Photography Perspective Composition: Towards Aesthetic Perspective Recommendation
> **Mode:** Run Results &nbsp;·&nbsp; **Generated:** 2026-06-19

> **Repository:** [https://github.com/vivoCameraResearch/p-p-c](https://github.com/vivoCameraResearch/p-p-c)

## 0. Runtime & GPU Check

In [ ]:
import os, sys, subprocess

# ── Detect runtime ────────────────────────────────────────────────────
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
    print("Runtime: Google Colab")
    # Uncomment to mount Google Drive for persistence:
    # from google.colab import drive
    # drive.mount('/content/drive')
except ImportError:
    print("Runtime: Non-Colab (rented GPU / local)")

# ── GPU check ────────────────────────────────────────────────────────
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("⚠️  No GPU detected — training will be slow on CPU.")

print(f"Python: {sys.version.split()[0]}")

## 1. Install Dependencies

In [ ]:
!pip install -q torch>=2.5.0 torchvision transformers>=4.45.0 huggingface_hub peft lightning pandas numpy matplotlib tqdm requests opencv-python imageio imageio-ffmpeg qwen-vl-utils accelerate

## 2. Clone Repository

In [ ]:
import os
if not os.path.exists('p-p-c'):
    !git clone https://github.com/vivoCameraResearch/p-p-c
os.chdir('p-p-c')
print('Repo ready.')

## 3. Download Utilities

Parallel multi-threaded downloader — reuse for datasets, weights, or any large files.

In [ ]:
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def download_file(url: str, dest, chunk_size: int = 1 << 20):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        return dest                           # skip if already on disk
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(dest, 'wb') as f, tqdm(
        desc=dest.name, total=total, unit='B',
        unit_scale=True, leave=False
    ) as bar:
        for chunk in r.iter_content(chunk_size):
            f.write(chunk)
            bar.update(len(chunk))
    return dest

def parallel_download(file_list: list, max_workers: int = 8):
    """file_list: list of (url, dest_path) tuples."""
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(download_file, url, dest): dest
                   for url, dest in file_list}
        for fut in tqdm(as_completed(futures), total=len(futures),
                        desc='Downloading', unit='file'):
            dest = futures[fut]
            try:
                fut.result()
                print(f"  ✓  {Path(dest).name}")
            except Exception as e:
                print(f"  ✗  {Path(dest).name}: {e}")

# Alternative: HuggingFace Hub download (parallel internally)
# !pip install -q huggingface_hub
# from huggingface_hub import snapshot_download
# snapshot_download(repo_id="LujianYao/PPC", local_dir="weights/")

## 4. Dataset Download

In [ ]:
# Specify dataset files:
# DATASET_FILES = [
#     ("https://...", "data/train.zip"),
#     ("https://...", "data/val.zip"),
# ]
# parallel_download(DATASET_FILES, max_workers=8)

## 5. Download Pretrained Weights

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id='LujianYao/PPC', local_dir='weights/')
print('Weights ready.')

## 6. Environment Configuration

### Sub-repositories required

PPC relies on three external sub-projects:

| Component | Purpose | Repository |
|---|---|---|
| ViewCrafter | 3D scene reconstruction + novel-view synthesis | [Drexubery/ViewCrafter](https://github.com/Drexubery/ViewCrafter) |
| DiffSynth-Studio | Wan2.1 I2V fine-tuning backbone | [modelscope/DiffSynth-Studio](https://github.com/modelscope/DiffSynth-Studio) |
| VideoAlign | PQA reward model (VideoReward) | [KwaiVGI/VideoAlign](https://github.com/KwaiVGI/VideoAlign) |

Run the cells below to clone and install each one.

In [ ]:
import os, subprocess, sys

def run(cmd, **kw):
    """Run a shell command and print output."""
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout: print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result.returncode

# ── 1. Clone VideoAlign (PQA / VideoReward) ────────────────────────────────
if not os.path.exists('VideoAlign'):
    run('git clone https://github.com/KwaiVGI/VideoAlign.git')
else:
    print('VideoAlign already cloned.')

# ── 2. Clone DiffSynth-Studio (Wan2.1 I2V backbone) ───────────────────────
if not os.path.exists('DiffSynth-Studio'):
    run('git clone https://github.com/modelscope/DiffSynth-Studio.git')
else:
    print('DiffSynth-Studio already cloned.')

# ── 3. Clone ViewCrafter (3D reconstruction) ──────────────────────────────
if not os.path.exists('ViewCrafter'):
    run('git clone https://github.com/Drexubery/ViewCrafter.git')
else:
    print('ViewCrafter already cloned.')

print('\nAll repositories cloned.')

## 7. Install Sub-repository Dependencies

Install dependencies for each sub-repository. **VideoAlign** needs `flash-attn` (may take ~5 min to compile on Colab). `DiffSynth-Studio` is installed in editable mode.

In [ ]:
import subprocess, sys

def pip(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print('ERROR:', result.stderr[-500:])
    else:
        print(f'Installed: {" ".join(args[:3])}...')
    return result.returncode

# DiffSynth-Studio (editable install)
print('Installing DiffSynth-Studio...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', 'DiffSynth-Studio'],
    capture_output=True, text=True
)
print('Done.' if result.returncode == 0 else f'ERROR: {result.stderr[-400:]}')

# peft, lightning, pandas for training/inference
pip('peft', 'lightning', 'pandas')

# flash-attn for VideoAlign (PQA model)
# NOTE: This may take several minutes on Colab due to compilation
print('Installing flash-attn 2.5.8 (may take a few minutes)...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'flash-attn==2.5.8', '--no-build-isolation'],
    capture_output=True, text=True, timeout=600
)
if result.returncode == 0:
    print('flash-attn installed successfully.')
else:
    print('flash-attn install failed — falling back to SDPA attention.')
    print('STDERR:', result.stderr[-400:])

## 8. Download Pretrained Weights

### Weights to download

| Model | HuggingFace ID | Size | Purpose |
|---|---|---|---|
| PPC LoRA checkpoint | `LujianYao/PPC` | ~307 MB | LoRA adapter for Wan2.1 I2V fine-tuned on PPC data |
| Wan2.1-I2V-14B-480P | `Wan-AI/Wan2.1-I2V-14B-480P` | ~28 GB | Base I2V model (large — use `ignore_patterns` to skip unneeded files) |
| VideoReward (PQA) | `KwaiVGI/VideoReward` | ~4 GB | Qwen2-VL-2B reward model for VQ/MQ/CA scoring |

**Note:** Wan2.1 is 28 GB. On Colab free tier, download only the essential shards. On a rented A100/H100 instance this is fine.

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
import os

# ── PPC LoRA checkpoint (307 MB) ───────────────────────────────────────────
print('Downloading PPC LoRA checkpoint...')
if not os.path.exists('weights/ppc_lora'):
    snapshot_download(
        repo_id='LujianYao/PPC',
        local_dir='weights/ppc_lora',
        ignore_patterns=['*.git*', '*.gitattributes']
    )
    print('PPC LoRA weights saved to weights/ppc_lora/')
else:
    print('PPC LoRA weights already present.')

# ── VideoReward / PQA model (KwaiVGI/VideoReward ~4 GB) ───────────────────
print('\nDownloading VideoReward PQA model...')
if not os.path.exists('VideoAlign/checkpoints/VideoReward'):
    os.makedirs('VideoAlign/checkpoints', exist_ok=True)
    snapshot_download(
        repo_id='KwaiVGI/VideoReward',
        local_dir='VideoAlign/checkpoints/VideoReward',
        ignore_patterns=['*.git*']
    )
    print('VideoReward saved to VideoAlign/checkpoints/VideoReward/')
else:
    print('VideoReward already present.')

## 9. Download Wan2.1-I2V-14B-480P Base Model

The Wan2.1 I2V backbone is ~28 GB. This cell downloads it via `snapshot_download` with `ignore_patterns` to skip redundant shards. Estimated time: 15–30 min on a fast GPU instance.

In [ ]:
from huggingface_hub import snapshot_download
import os

WAN_LOCAL = 'models/Wan-AI/Wan2.1-I2V-14B-480P'

if not os.path.exists(WAN_LOCAL):
    os.makedirs(WAN_LOCAL, exist_ok=True)
    print('Downloading Wan2.1-I2V-14B-480P (~28 GB). This will take time...')
    snapshot_download(
        repo_id='Wan-AI/Wan2.1-I2V-14B-480P',
        local_dir=WAN_LOCAL,
        ignore_patterns=['*.git*', 'figures/*', '*.md'],
    )
    print(f'Wan2.1 model saved to {WAN_LOCAL}/')
else:
    print(f'Wan2.1 model already present at {WAN_LOCAL}/')

# List key files
for f in sorted(os.listdir(WAN_LOCAL))[:15]:
    print(' ', f)

## 10. Load PQA (VideoReward) Reward Model

### PQA = VideoReward fine-tuned on VQ / MQ / CA

The `VideoReward` model is a fine-tuned Qwen2-VL-2B that scores a video across three axes reported in Table 2 of the paper:
- **VQ** — Visual Quality (context-agnostic clarity / aesthetics)
- **MQ** — Motion Quality (smoothness, consistency)
- **CA** — Composition Aesthetic (compositional improvement across the transformation)

We use the VideoAlign `inference.py` interface with a slight modification to expose all three score dimensions.

In [ ]:
import sys, os
sys.path.insert(0, 'VideoAlign')

import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

VIDEO_REWARD_CKPT = 'VideoAlign/checkpoints/VideoReward'

print('Loading VideoReward (PQA) model...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

try:
    # Load processor and model
    processor = AutoProcessor.from_pretrained(
        VIDEO_REWARD_CKPT,
        trust_remote_code=True
    )
    pqa_model = Qwen2VLForConditionalGeneration.from_pretrained(
        VIDEO_REWARD_CKPT,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True
    )
    pqa_model.eval()
    print('PQA model loaded successfully.')
except Exception as e:
    print(f'Error loading PQA model: {e}')
    print('Hint: Check that VideoAlign/checkpoints/VideoReward exists and is non-empty.')

## 11. Load PPC LoRA Model (Wan2.1 + PPC fine-tune)

### PPC inference model = Wan2.1-I2V-14B-480P + PPC LoRA

The PPC LoRA adapter (`LujianYao/PPC` → `wan_lora_example.ckpt`, 307 MB) is loaded on top of the Wan2.1 I2V base model via `peft`. This is the model that generates the perspective-transformation video from an input image.

In [ ]:
import sys, os, torch
sys.path.insert(0, 'DiffSynth-Studio')

from diffsynth import ModelManager, WanVideoPipeline
from peft import PeftModel

WAN_LOCAL  = 'models/Wan-AI/Wan2.1-I2V-14B-480P'
LORA_CKPT  = 'weights/ppc_lora/wan_lora_example.ckpt'

print('Loading Wan2.1 I2V base model (this may take ~2 min)...')
try:
    model_manager = ModelManager(
        torch_dtype=torch.bfloat16,
        device='cuda'
    )
    model_manager.load_models(
        [
            os.path.join(WAN_LOCAL, 'models_t5_umt5-xxl-enc-bf16.pth'),
            os.path.join(WAN_LOCAL, 'Wan2.1_VAE.pth'),
            os.path.join(WAN_LOCAL, 'Wan2.1-I2V-14B-480P'),  # DiT weights
        ],
        torch_dtype=torch.bfloat16,
    )
    pipe = WanVideoPipeline.from_model_manager(model_manager)
    print('Base model loaded.')
except Exception as e:
    print(f'Error loading base model: {e}')
    pipe = None

# Apply PPC LoRA adapter
if pipe is not None and os.path.exists(LORA_CKPT):
    try:
        # Load LoRA state dict
        lora_state = torch.load(LORA_CKPT, map_location='cpu')
        # Apply via peft or directly (depends on checkpoint format)
        pipe.dit.load_state_dict(lora_state, strict=False)
        print(f'PPC LoRA applied from {LORA_CKPT}')
    except Exception as e:
        print(f'LoRA load warning: {e}')
        print('Attempting via DiffSynth lora loader...')
        try:
            pipe.load_lora(LORA_CKPT, lora_alpha=1.0)
            print('LoRA applied via DiffSynth loader.')
        except Exception as e2:
            print(f'LoRA load failed: {e2}')
else:
    if pipe is None:
        print('Skipping LoRA — base model not loaded.')
    else:
        print(f'LoRA checkpoint not found at {LORA_CKPT}. Check weights download.')

## 12. Inference: Generate PPC Video from Input Image

### Single-image → perspective-transformation video

Given an input image `s` (suboptimal perspective), the PPC model samples:
$$v \sim p_\theta(v \mid s)$$
producing a video that transitions from `s` (less-favorable perspective) to an aesthetically improved viewpoint.

**Key inference parameters (from paper):**
- Video resolution: 480p (480 × 832)
- Num frames: 81 (≈ 3 s at ~27 fps)
- No text prompt required — image-conditioned only
- Guidance box is derived from the last frame via homography

**Usage:** Replace `INPUT_IMAGE_PATH` with your own image or use one of the examples in `p-p-c/figures/`.

In [ ]:
import torch
from PIL import Image
import numpy as np
import os

INPUT_IMAGE_PATH = 'p-p-c/figures/first.png'  # Replace with your own image
OUTPUT_VIDEO_PATH = 'output/ppc_result.mp4'
os.makedirs('output', exist_ok=True)

if pipe is None:
    print('Model not loaded — skipping inference.')
elif not os.path.exists(INPUT_IMAGE_PATH):
    print(f'Input image not found: {INPUT_IMAGE_PATH}')
    print('Place your test image at that path, or update INPUT_IMAGE_PATH.')
else:
    input_image = Image.open(INPUT_IMAGE_PATH).convert('RGB')
    print(f'Input image: {input_image.size} (W×H)')

    # Resize to 480p (832×480) — WxH, landscape
    input_image = input_image.resize((832, 480), Image.LANCZOS)

    print('Running PPC inference (81 frames @ 480p)...')
    with torch.inference_mode():
        video_frames = pipe(
            input_image=input_image,
            prompt='',          # no text prompt needed for PPC
            negative_prompt='',
            num_inference_steps=50,
            num_frames=81,
            height=480,
            width=832,
            guidance_scale=5.0,
        )

    # Save as MP4
    import imageio
    writer = imageio.get_writer(OUTPUT_VIDEO_PATH, fps=27, quality=8)
    for frame in video_frames:
        writer.append_data(np.array(frame))
    writer.close()
    print(f'Video saved to {OUTPUT_VIDEO_PATH}')

## 13. PQA Scoring: Evaluate the Generated Video

### Reproducing PQA scores (VQ, MQ, CA)

The paper reports three PQA dimensions for each I2V backbone (Table 2 of the paper):

| Model | VQ ↑ | MQ ↑ | CA ↑ |
|---|---|---|---|
| CogVideoX 1.5 5B | 0.7073 | 0.7311 | **0.7196** |
| HunYuan I2V | **0.7216** | **0.7496** | 0.7070 |
| Wan2.1 14B | 0.7195 | 0.7454 | 0.7072 |

Here we score our generated video with the VideoReward PQA model to get VQ, MQ, CA scores.

In [ ]:
import sys, os, torch
import numpy as np
from pathlib import Path

OUTPUT_VIDEO_PATH = 'output/ppc_result.mp4'

if not os.path.exists(OUTPUT_VIDEO_PATH):
    print(f'Output video not found at {OUTPUT_VIDEO_PATH}. Run inference cell first.')
elif 'pqa_model' not in dir() or pqa_model is None:
    print('PQA model not loaded. Run the PQA model loading cell first.')
else:
    # Use VideoAlign inference helper
    sys.path.insert(0, 'VideoAlign')
    try:
        from inference import score_video
        scores = score_video(pqa_model, processor, OUTPUT_VIDEO_PATH, device='cuda')
        print('PQA Scores for generated video:')
        print(f'  VQ (Visual Quality)       : {scores.get("VQ", "N/A"):.4f}')
        print(f'  MQ (Motion Quality)       : {scores.get("MQ", "N/A"):.4f}')
        print(f'  CA (Composition Aesthetic): {scores.get("CA", "N/A"):.4f}')
    except ImportError:
        # Fallback: direct model call using Qwen2-VL interface
        print('Using direct Qwen2-VL scoring interface...')
        import imageio.v3 as iio
        frames = iio.imread(OUTPUT_VIDEO_PATH, plugin='pyav')  # shape: (T, H, W, C)
        T = frames.shape[0]
        # Sample 8 evenly-spaced frames for scoring (as per VideoReward protocol)
        idxs = np.linspace(0, T-1, min(8, T), dtype=int)
        sampled = [frames[i] for i in idxs]

        from qwen_vl_utils import process_vision_info
        from PIL import Image

        # Build VQ prompt
        def build_prompt(dimension: str) -> str:
            prompts = {
                'VQ': 'Evaluate the visual quality of this video. Score from 0 to 1 where 1 is the best.',
                'MQ': 'Evaluate the motion quality and smoothness of camera movement in this video. Score from 0 to 1.',
                'CA': 'Evaluate how well this video improves the photographic composition aesthetic from start to end. Score from 0 to 1.',
            }
            return prompts[dimension]

        scores = {}
        for dim in ['VQ', 'MQ', 'CA']:
            messages = [{
                'role': 'user',
                'content': [
                    {'type': 'video', 'video': [Image.fromarray(f) for f in sampled],
                     'fps': 1.0},
                    {'type': 'text', 'text': build_prompt(dim)},
                ]
            }]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors='pt'
            ).to(pqa_model.device)

            with torch.inference_mode():
                output = pqa_model(**inputs, return_dict=True)
                # Extract reward score from last token logit (VideoReward protocol)
                score = torch.sigmoid(output.logits[:, -1, :]).mean().item()
            scores[dim] = score
            print(f'  {dim}: {score:.4f}')

## 14. Results vs Paper (Table 2 Comparison)

Compare our reproduced scores with the numbers reported in **Table 2** of the paper.

In [ ]:
import pandas as pd

# Paper-reported results (Table 2)
paper_results = {
    'Model': ['CogVideoX 1.5 5B', 'HunYuan I2V', 'Wan2.1 14B (Paper)'],
    'CMM ↑':  [0.5501, 0.4928, 0.5989],
    'FVD ↓':  [303,    264,    345],
    'PSNR ↑': [8.2380, 9.4017, 9.3668],
    'SSIM ↑': [0.2611, 0.3537, 0.3265],
    'LPIPS ↓':[0.7969, 0.7915, 0.7808],
    'VQ ↑':   [0.7073, 0.7216, 0.7195],
    'MQ ↑':   [0.7311, 0.7496, 0.7454],
    'CA ↑':   [0.7196, 0.7070, 0.7072],
}
df_paper = pd.DataFrame(paper_results)

print('=== Paper Reported Results (Table 2) ===')
print(df_paper.to_string(index=False))
print()

# RLHF improvement (Table 4, HunYuan + RLHF)
rlhf_results = {
    'Setting': ['HunYuan w/o RLHF', 'HunYuan + RLHF (Flow-DPO)'],
    'CMM ↑':  [0.4928, 0.5014],
    'FVD ↓':  [264.77, 270.22],
    'VQ ↑':   [0.7216, 0.7477],
    'MQ ↑':   [0.7496, 0.7774],
    'CA ↑':   [0.7070, 0.7342],
}
df_rlhf = pd.DataFrame(rlhf_results)
print('=== RLHF Ablation (Table 4) ===')
print(df_rlhf.to_string(index=False))
print()

# If inference was run, append our scores
try:
    our_row = {
        'Model': 'Wan2.1 14B + PPC LoRA (Reproduced)',
        'CMM ↑': float('nan'),
        'FVD ↓': float('nan'),
        'PSNR ↑': float('nan'),
        'SSIM ↑': float('nan'),
        'LPIPS ↓': float('nan'),
        'VQ ↑': scores.get('VQ', float('nan')),
        'MQ ↑': scores.get('MQ', float('nan')),
        'CA ↑': scores.get('CA', float('nan')),
    }
    df_ours = pd.DataFrame([our_row])
    df_combined = pd.concat([df_paper, df_ours], ignore_index=True)
    print('=== Paper Results + Our Reproduction ===')
    print(df_combined.to_string(index=False))
except NameError:
    print('(Run inference and PQA scoring cells first to see comparison.)')

## ⚠️ Manual Steps Required

1. Flash-attn 2.5.8 requires CUDA 11.8+ and torch >=2.0. If compilation fails on Colab, re-run the install cell or skip flash-attn — VideoAlign will fall back to SDPA attention.
2. The Wan2.1-I2V-14B-480P model is ~28 GB. On Colab free tier (15 GB disk) you may need to use a rented A100/H100 instance (RunPod, Vast.ai, Lambda Labs). Alternatively, stream shards selectively.
3. DUSt3R geometry prior for ViewCrafter: download from https://download.europe.naverlabs.com/ComputerVision/DUSt3R/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth and place in ViewCrafter/checkpoints/
4. The PPC LoRA checkpoint at LujianYao/PPC contains wan_lora_example.ckpt. If the exact loading API for DiffSynth-Studio changes, check the DiffSynth-Studio docs for the correct lora load method.
5. For full CMM/FVD/PSNR/SSIM/LPIPS evaluation (Table 2), you need the PPC test set. The paper does not publicly release the test set — contact the authors at the project page: https://vivocameraresearch.github.io/ppc/